# 문항 1 네이버 주식 Raw 적재와 Clean 정제

In [ ]:
USE fsc_db;

CREATE TABLE IF NOT EXISTS tb_nf_stock (
    bas_dt DATE NOT NULL,
    srtn_cd CHAR(6) NOT NULL,
    itms_nm VARCHAR(50),
    clpr BIGINT,
    vs BIGINT,
    mkp BIGINT,
    hipr BIGINT,
    lopr BIGINT,
    trqu BIGINT,
    raw_id BIGINT,
    PRIMARY KEY (bas_dt, srtn_cd)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;

CREATE TABLE IF NOT EXISTS tb_fsc_stock (
    bas_dt DATE NOT NULL,
    srtn_cd CHAR(6) NOT NULL,
    itms_nm VARCHAR(50),
    clpr BIGINT,
    vs BIGINT,
    mkp BIGINT,
    hipr BIGINT,
    lopr BIGINT,
    trqu BIGINT,
    raw_id BIGINT,
    PRIMARY KEY (bas_dt, srtn_cd)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;

종목명 테이블을 만든후 따로 저장해 코드를 사용해 조인해서 사용하면 좋을 것 같습니다.

In [ ]:
# 네이버 테이블

import requests
from bs4 import BeautifulSoup
import re
import pymysql
import os
from pymysql.cursors import DictCursor
from dotenv import load_dotenv
from datetime import datetime
import pandas as pd

load_dotenv()

URL = "https://finance.naver.com/item/sise_day.naver"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",

}

BATCH_SIZE = 50

DB_CONFIG = {
    'host': os.getenv('DB_HOST'),
    'port': int(os.getenv('DB_PORT')),
    'user': os.getenv('DB_USER'),
    'password': os.getenv('DB_PASSWORD'),
    'database': os.getenv('DB_NAME'),
    'charset': 'utf8mb4',
    'cursorclass': DictCursor
}

def connect():
    return pymysql.connect(**DB_CONFIG)

def insert_data(data):
    conn = connect()
    try:
        INSERT_QUERY = """
            INSERT INTO tb_nf_stock (bas_dt, srtn_cd, itms_nm, clpr, vs, mkp, hipr, lopr, trqu, raw_id)
            VALUES (%(날짜)s, 035420, NULL, %(종가)s, %(전일비)s, %(시가)s, %(고가)s, %(저가)s, %(거래량)s, NULL)
            ON DUPLICATE KEY UPDATE
                clpr = VALUES(clpr),
                vs = VALUES(vs),
                mkp = VALUES(mkp),
                hipr = VALUES(hipr),
                lopr = VALUES(lopr),
                trqu = VALUES(trqu)
        """
        for i in range(0, len(data), BATCH_SIZE):
            with conn.cursor() as cur:
                cur.executemany(INSERT_QUERY, data[i:i+BATCH_SIZE])
            conn.commit()
            print("적재%d / %d" % (min(i + BATCH_SIZE, len(data)), len(data)))
    except Exception as e:
        print(e)
    finally:
        conn.close()

result = []
count = 0
# 365일 이므로 38 사용
for i in range(1, 38):
    # 1년치 데이터 수집후 멈추기.
    if count >= 365:
        break

    PARAMS = {
        "code" : "005930",
        "page" : i
    }

    try:
        response = requests.get(URL, params=PARAMS, headers=HEADERS)
        if response.status_code != 200:
            raise Exception(f"Error: {response.status_code}")
    except:
        print("Request had error")
        break
    soup = BeautifulSoup(response.text, "html.parser")
    trs = soup.select('table.type2 tr[onmouseover="mouseOver(this)"]')
    for tr in trs:
        # 1년치 데이터 수집후 멈추기.
        if count >= 365:
            break

        date = tr.select_one("td:nth-child(1)").text.strip()
        formatted_date = datetime.strptime(date, "%Y.%m.%d").date()
        
        change = tr.select_one("span.blind").text.strip()
        amount = tr.select("td")[2].text.strip()
        numeric_amount = re.sub(r'[\D]', '', amount)
        if change == "상승":
            change = ""
        elif change == "하락":
            change = "-"
        else:
            change = ""

        new_change = change + numeric_amount
        result.append({
            "날짜" : formatted_date,
            "종가" : int(tr.select_one("td:nth-child(2)").text.strip().replace(",", "")),
            "전일비" : int(new_change),
            "시가" : int(tr.select_one("td:nth-child(4)").text.strip().replace(",", "")),
            "고가" : int(tr.select_one("td:nth-child(5)").text.strip().replace(",", "")),
            "저가" : int(tr.select_one("td:nth-child(6)").text.strip().replace(",", "")),
            "거래량" : int(tr.select_one("td:nth-child(7)").text.strip().replace(",", "")),
        })
        count += 1

insert_data(result)

query = """SELECT * FROM tb_nf_stock"""

with connect().cursor() as cur:
    cur.execute(query)
    rows = cur.fetchall()

df_nf = pd.DataFrame(rows)
col_len = len(df_nf.columns)
dupe = df_nf.duplicated(['bas_dt', 'srtn_cd']).mean()
min_date = df_nf['bas_dt'].min()
max_date = df_nf['bas_dt'].max()

print("기간: " + str(min_date) + " ~ " + str(max_date))
print("종목 수:", col_len)
print("중복률:", dupe)
# 중복없으니 중복 제거 안함

적재50 / 365
적재100 / 365
적재150 / 365
적재200 / 365
적재250 / 365
적재300 / 365
적재350 / 365
적재365 / 365
기간: 2025-03-04 ~ 2026-08-28
종목 수: 10
중복률: 0.0


In [175]:
# 금융위 주식용 테이블

import pandas as pd
import pymysql
import os
from pymysql.cursors import DictCursor
from dotenv import load_dotenv
from datetime import datetime
import json
import ast

load_dotenv()

DB_CONFIG = {
    'host': os.getenv('DB_HOST'),
    'port': int(os.getenv('DB_PORT')),
    'user': os.getenv('DB_USER'),
    'password': os.getenv('DB_PASSWORD'),
    'database': os.getenv('DB_NAME'),
    'charset': 'utf8mb4',
    'cursorclass': DictCursor
}

BATCH_SIZE = 50

def connect():
    return pymysql.connect(**DB_CONFIG)

def count_raw(conn):
    cur = conn.cursor()
    cur.execute('SELECT COUNT(*) AS cnt FROM raw_item')
    total_count = cur.fetchone()['cnt']
    cur.close()
    return total_count

def get_raw_items(conn, i, size):
    cur = conn.cursor()
    cur.execute(
        'SELECT * FROM raw_item WHERE raw_id > %s LIMIT %s', 
        (i, size)
    )
    datas = cur.fetchall()
    cur.close()
    return datas

def insert_data(data):
    conn = connect()
    try:
        INSERT_QUERY = """
            INSERT INTO tb_fsc_stock (bas_dt, srtn_cd, itms_nm, clpr, vs, mkp, hipr, lopr, trqu)
            VALUES (%(basDt)s, %(srtnCd)s, %(itmsNm)s, %(clpr)s, %(vs)s, %(mkp)s, %(hipr)s, %(lopr)s, %(trqu)s)
            ON DUPLICATE KEY UPDATE
                clpr = VALUES(clpr),
                vs = VALUES(vs),
                mkp = VALUES(mkp),
                hipr = VALUES(hipr),
                lopr = VALUES(lopr),
                trqu = VALUES(trqu)
        """
        for i in range(0, len(data), BATCH_SIZE):
            with conn.cursor() as cur:
                cur.executemany(INSERT_QUERY, data[i:i+BATCH_SIZE])
            conn.commit()
            print("적재%d / %d" % (min(i + BATCH_SIZE, len(data)), len(data)))
    except Exception as e:
        print(e)
    finally:
        conn.close()

# 전체 개수 확인
conn = connect()
total_count = count_raw(conn)
conn.close()
print('총 raw item 수:', total_count)

result = []

for i in range(0, total_count, BATCH_SIZE):
    try:
        conn = connect()
        datas = get_raw_items(conn, i, BATCH_SIZE)
        df = pd.DataFrame(datas)
        temp = df['payload']
        for item in temp:
            result.append(ast.literal_eval(item))

    except Exception as e:
        print('알 수 없는 오류 발생', e, i)
    finally:
        conn.close()

for i in result:
    i['basDt'] = datetime.strptime(i['basDt'], '%Y%m%d').date()
insert_data(result)

query = """SELECT * FROM tb_fsc_stock"""

with connect().cursor() as cur:
    cur.execute(query)
    rows = cur.fetchall()

df_fsc = pd.DataFrame(rows)
col_len = len(df_fsc.columns)
dupe = df_fsc.duplicated(['bas_dt', 'srtn_cd']).mean()
min_date = df_fsc['bas_dt'].min()
max_date = df_fsc['bas_dt'].max()

print("기간: " + str(min_date) + " ~ " + str(max_date))
print("종목 수:", col_len)
print("중복률:", dupe)
# 중복없으니 중복 제거 안함

총 raw item 수: 100
적재50 / 100
적재100 / 100
기간: 2025-12-16 ~ 2025-12-30
종목 수: 10
중복률: 0.0


# 문항 2 분석 지표 설계와 Mart 적재

In [176]:
# (1) tb_mart_stock_monthly — 종목 × 월 집계
df_first_qn = df_fsc.copy()
df_first_qn['bas_dt'] = pd.to_datetime(df_first_qn['bas_dt'])

tb_mart_stock_monthly = (df_first_qn.assign(bas_dt=df_first_qn['bas_dt'].dt.to_period('M'))
        .groupby(["srtn_cd", "bas_dt"], as_index=False)
           .agg(avg_clpr=("clpr", "mean"),
                max_clpr=("clpr", "max"),
                min_clpr=("clpr", "min"),
                open_clpr=("clpr", "first"),      
                close_clpr=("clpr", "last"), 
                sum_trqu=("trqu", "sum"),
                avg_trqu=("trqu", "mean"),
                total_days=("bas_dt", "count")))

# 10일 미만 제외
tb_mart_stock_monthly = tb_mart_stock_monthly[tb_mart_stock_monthly['total_days'] >= 10]

In [194]:
from sqlalchemy import create_engine

host = os.getenv('DB_HOST')
port = 3306
password = os.getenv('DB_PASSWORD')
db = os.getenv('DB_NAME')

engine = create_engine(f'mysql+pymysql://root:{password}@{host}:{port}/{db}')

tb_mart_stock_monthly


,srtn_cd,bas_dt,avg_clpr,max_clpr,min_clpr,open_clpr,close_clpr,sum_trqu,avg_trqu,total_days
0,000660,2025-12,582200.0,651000,530000,530000,651000,31944339,3194433.9,10
1,005380,2025-12,288400.0,296500,282500,286000,296500,8712174,871217.4,10
2,005930,2025-12,111410.0,119900,102800,102800,119900,218724243,21872424.3,10
3,006400,2025-12,280800.0,295000,269500,293500,269500,4014418,401441.8,10
4,035420,2025-12,235600.0,242500,230500,231500,242500,8920516,892051.6,10
5,035720,2025-12,58710.0,61200,57000,57900,60100,18701689,1870168.9,10
6,051910,2025-12,343450.0,364000,333000,354000,333000,2110453,211045.3,10
7,055550,2025-12,77020.0,78600,75800,75800,76900,9228396,922839.6,10
8,068270,2025-12,182890.0,185600,180000,185600,181000,3809396,380939.6,10
9,105560,2025-12,125070.0,126500,123700,123700,124700,5911570,591157.0,10


In [195]:
tb_mart_stock_monthly.to_sql('tb_mart_stock_monthly', engine, if_exists='replace', index=False)

10

In [ ]:
# (2) tb_mart_stock_daily — 파생 지표 추가 테이블
df_second_qn = df_fsc.copy()

df_second_qn.sort_values(by=['srtn_cd', 'bas_dt'], ascending=True, inplace=True)

df_second_qn['chg_pct'] = (df_second_qn.groupby('srtn_cd')["trqu"].pct_change(fill_method=None)*100).round(2)

df_second_qn['ma5'] = df_second_qn.groupby('srtn_cd')['trqu'].transform(lambda x: x.rolling(5).mean()).round(1)
df_second_qn['ma20'] = df_second_qn.groupby('srtn_cd')['trqu'].transform(lambda x: x.rolling(20).mean()).round(1)

df_second_qn['vol_ratio'] = (df_second_qn['trqu'] / df_second_qn['ma20']).round(1)

df_second_qn

,bas_dt,srtn_cd,itms_nm,clpr,vs,mkp,hipr,lopr,trqu,raw_id,chg_pct,ma5,ma20,vol_ratio
0,2025-12-16,000660,SK하이닉스,530000,-24000,545000,551000,530000,3287998,None,NaN,NaN,NaN,NaN
10,2025-12-17,000660,SK하이닉스,551000,21000,530000,554000,525000,2692626,None,-18.11,NaN,NaN,NaN
20,2025-12-18,000660,SK하이닉스,552000,1000,538000,563000,537000,2959059,None,9.89,NaN,NaN,NaN
30,2025-12-19,000660,SK하이닉스,547000,-5000,569000,570000,547000,3440005,None,16.25,NaN,NaN,NaN
40,2025-12-22,000660,SK하이닉스,580000,33000,574000,580000,567000,2878464,None,-16.32,3051630.4,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,2025-12-23,105560,KB금융,126000,-500,126800,127300,125300,484510,None,-7.02,646345.0,NaN,NaN
69,2025-12-24,105560,KB금융,126100,100,125700,126600,125300,318252,None,-34.31,547110.6,NaN,NaN
79,2025-12-26,105560,KB금융,124600,-1500,126000,126100,124300,329428,None,3.51,475905.0,NaN,NaN
89,2025-12-29,105560,KB금융,125600,1000,122600,125600,121400,462846,None,40.50,423224.6,NaN,NaN
